In [ ]:
# Full spacer nucleotide + full spacer MFE structure + 20 nt target nucleotide + 20 nt target paired structure

# New Feature #54-#95 for CNN-still every base pairing state in a 2D array
# "0" means the base at this position is unpaired, "1" is paired
# The probability of this structure is reflected as the frequency of apperance of thsi structure in the array. 
# For example, if we sample 100 structures in a 100*42 array and the frequency of Struct A is 85%, 85 rows in this array will be this structure. 
# 4 channels, unpaire (1, 0, 0, 0); A/T paired (0, 1, 0, 0); G/C paired (0, 0, 1, 0); G/U paired (0, 0, 0, 1)
# Not considering back bracket

# Only consider MFE structure
# 8 channels, first elements represent base types and the last element represent pairing state, one additional channel encoding guide_seq on spacer 
# PAM considered
# Repeat domain after hybridization considered
# Top10 suboptimal structures considered
# Prob of guide and target suboptimal structures considered
# Considering the free energy of most consecutive base pairs in spacer and target
# Considering 5' and 3' overhang of spacer and target
# Considering 5' and 3' hybridized bases
# Considering the target free energy segment (seed region, middle region, distal region)

from nupack import *
import RNA
import math
import itertools
import numpy as np

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))

def RNA_to_DNA(RNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'U': 'T'}
    return ''.join(match.get(base, base) for base in (RNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def find_parentheses_pairs(s):
    stack = []  # Stack to keep track of '(' positions
    pairs = {}  # Dictionary to store pairs of indices (key is '(' or ')', value is the paired parenthesis index)
    
    # Iterate through the string with index
    for i, char in enumerate(s):
        if char == '(':
            stack.append(i)  # Push the index of '(' onto the stack
        elif char == ')':
            if stack:  # Ensure stack is not empty
                open_index = stack.pop()  # Pop the index of the matching '('
                pairs[open_index] = i  # Store the pair (open_index, close_index)
                pairs[i] = open_index  # Store the reverse pair (close_index, open_index)
    
    return pairs

def get_outermost_paired_parenthesis(pairs, position):
    # If position is an opening parenthesis, we find its outermost pair
    if position in pairs:
        # Return the paired parenthesis
        return pairs[position]
    else:
        return None  # If no matching parenthesis found for the given position

def normalize_guide(value):
    normalized_value = (value - (-25.21)) / (0 - (-25.21))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-48.98)) / (-13.23 - (-48.98))
    return normalized_value

def normalize_ssDNA_target_bh(value):
    normalized_value = (value - (-12.20)) / (0 - (-12.20))
    return normalized_value



def normalize_guide_conse(value):
    normalized_value = (value - (-52.06)) / (2.61 - (-52.06))
    return normalized_value

def normalize_target_conse(value):
    normalized_value = (value - (-40.52)) / (0 - (-40.52))
    return normalized_value

def normalize_ssDNA_target_bh_conse(value):
    normalized_value = (value - (-15.21)) / (0 - (-15.21))
    return normalized_value



def normalize_guide_conse_unpaired(value):
    normalized_value = (value - (-52.06)) / (2.61 - (-52.06))
    return normalized_value

def normalize_target_conse_unpaired(value):
    normalized_value = (value - (-40.52)) / (0 - (-40.52))
    return normalized_value

def normalize_ssDNA_target_bh_conse_unpaired(value):
    normalized_value = (value - (-33.55)) / (-0.32 - (-33.55))
    return normalized_value



def normalize_guide_overhang(value):
    normalized_value = (value - (-52.06)) / (2.61 - (-52.06))
    return normalized_value

def normalize_target_overhang(value):
    normalized_value = (value - (-40.52)) / (0 - (-40.52))
    return normalized_value

def normalize_ssDNA_target_bh_overhang(value):
    normalized_value = (value - (-33.55)) / (0 - (-33.55))
    return normalized_value



def normalize_guide_paired(value):
    normalized_value = (value - (-52.06)) / (2.61 - (-52.06))
    return normalized_value

def normalize_target_paired(value):
    normalized_value = (value - (-40.52)) / (0 - (-40.52))
    return normalized_value

def normalize_ssDNA_target_bh_paired(value):
    normalized_value = (value - (-15.21)) / (0 - (-15.21))
    return normalized_value
    


def normalize_seed(value):
    normalized_value = (value - (-10.99)) / (-3.33 - (-10.99))
    return normalized_value

def normalize_middle(value):
    normalized_value = (value - (-13.07)) / (-4.12 - (-13.07))
    return normalized_value

def normalize_distal(value):
    normalized_value = (value - (-13.07)) / (-4.12 - (-13.07))
    return normalized_value


def normalize_target_conse_3(value):
    normalized_value = (value - (-3.89)) / (0.41 - (-3.89))
    return normalized_value

def normalize_target_conse_4(value):
    normalized_value = (value - (-7.20)) / (-0.80 - (-7.20))
    return normalized_value

def normalize_target_conse_5(value):
    normalized_value = (value - (-9.50)) / (-1.80 - (-9.50))
    return normalized_value

def normalize_target_conse_6(value):
    normalized_value = (value - (-12.80)) / (-3.00 - (-12.80))
    return normalized_value

def normalize_target_conse_7(value):
    normalized_value = (value - (-15.11)) / (-4.01 - (-15.11))
    return normalized_value

def normalize_target_conse_8(value):
    normalized_value = (value - (-18.41)) / (-5.21 - (-18.41))
    return normalized_value



def find_max_base_pairs(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '()':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def find_max_unpaired(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '.':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def detect_5_overhang(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_overhang(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def detect_5_paired(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_paired(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def is_all_parens(s):
    for char in s:
        if char not in ("(", ")"):
            return False
    return True
    

file_paths = [
    'Feature_guide_target_complete_value_features_120D_all_positive_HTCas9_unique_KD_reduced_feature_model.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  # No need to write anything, just open and close the file to delete its content

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('HTCas9_full_guide_sequences_unique.txt', 'r')    # Input crRNA sequences with AsCas12a direct repeat
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('HTCas9_target_sequences_noPAM_unique.txt', 'r')    # Input target sequences with AsCas12a direct repeat
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('HTCas9_target_sequences_noPAM_unique.txt', 'r')    # Input target sequences with AsCas12a direct repeat
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf
min_ssDNA_target_bh_energy = np.inf
max_ssDNA_target_bh_energy = -np.inf

min_guide_energy_conse = np.inf
max_guide_energy_conse = -np.inf
min_target_energy_conse = np.inf
max_target_energy_conse = -np.inf
min_ssDNA_target_bh_energy_conse = np.inf
max_ssDNA_target_bh_energy_conse = -np.inf

min_guide_energy_conse_unpaired = np.inf
max_guide_energy_conse_unpaired = -np.inf
min_target_energy_conse_unpaired = np.inf
max_target_energy_conse_unpaired = -np.inf
min_ssDNA_target_bh_energy_conse_unpaired = np.inf
max_ssDNA_target_bh_energy_conse_unpaired = -np.inf

min_guide_energy_PAM_proximal_overhang = np.inf
max_guide_energy_PAM_proximal_overhang = -np.inf
min_target_energy_PAM_proximal_overhang = np.inf
max_target_energy_PAM_proximal_overhang = -np.inf
min_ssDNA_target_bh_energy_PAM_proximal_overhang = np.inf
max_ssDNA_target_bh_energy_PAM_proximal_overhang = -np.inf

min_guide_energy_PAM_distal_overhang = np.inf
max_guide_energy_PAM_distal_overhang = -np.inf
min_target_energy_PAM_distal_overhang = np.inf
max_target_energy_PAM_distal_overhang = -np.inf
min_ssDNA_target_bh_energy_PAM_distal_overhang = np.inf
max_ssDNA_target_bh_energy_PAM_distal_overhang = -np.inf

min_guide_energy_PAM_proximal_paired = np.inf
max_guide_energy_PAM_proximal_paired = -np.inf
min_target_energy_PAM_proximal_paired = np.inf
max_target_energy_PAM_proximal_paired = -np.inf
min_ssDNA_target_bh_energy_PAM_proximal_paired = np.inf
max_ssDNA_target_bh_energy_PAM_proximal_paired = -np.inf

min_guide_energy_PAM_distal_paired = np.inf
max_guide_energy_PAM_distal_paired = -np.inf
min_target_energy_PAM_distal_paired = np.inf
max_target_energy_PAM_distal_paired = -np.inf
min_ssDNA_target_bh_energy_PAM_distal_paired = np.inf
max_ssDNA_target_bh_energy_PAM_distal_paired = -np.inf

max_seed_energy = -np.inf
min_seed_energy = np.inf

max_middle_energy = -np.inf
min_middle_energy = np.inf

max_distal_energy = -np.inf
min_distal_energy = np.inf


array_max_ensemble_energy_selected_bases_3 = []
array_min_ensemble_energy_selected_bases_3 = []

array_max_ensemble_energy_selected_bases_4 = []
array_min_ensemble_energy_selected_bases_4 = []

array_max_ensemble_energy_selected_bases_5 = []
array_min_ensemble_energy_selected_bases_5 = []

array_max_ensemble_energy_selected_bases_6 = []
array_min_ensemble_energy_selected_bases_6 = []

array_max_ensemble_energy_selected_bases_7 = []
array_min_ensemble_energy_selected_bases_7 = []

array_max_ensemble_energy_selected_bases_8 = []
array_min_ensemble_energy_selected_bases_8 = []



# Initialize struct_prob_array
struct_prob_array = []

for i in range (0, len(guide_array)):

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Initialize struct_prob_array
    struct_prob_array_guide = []
    struct_prob_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_DNA)
    ensemble_energy_target = partition_function_target[1]
    partition_function_ssDNA_target_bh = pfunc(strands=target_truncated, model=my_model_DNA)
    ensemble_energy_ssDNA_target_bh = partition_function_ssDNA_target_bh[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=0.01, model=my_model_RNA)  # The energy gap of 5.68 kcal/mol refers to minimal probablity of 0.01%
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=0.01, model=my_model_DNA)
    subopt_structures_ssDNA_target_bh = subopt(strands=target_truncated, energy_gap=0.01, model=my_model_DNA)

    # Compute the probability for each suboptimal structure using Boltzmann equilibrium probability distribution (ViennaRNA package)
    struct_prob_array_unit = []
    kT = RNA.exp_param().kT / 1000.
    prob_sub_guide = math.exp((ensemble_energy_guide - subopt_structures_guide[0].energy) / kT)
    pairs_guide = find_parentheses_pairs(str(subopt_structures_guide[0].structure))

    struct_prob_array_unit = []

    len_scaffold = 80
    len_spacer = 20
    
    # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy and subopt_structures_ssDNA_target_bh[0].energy
    prob_sub_guide = math.exp((ensemble_energy_guide - subopt_structures_guide[0].energy) / kT)
    if subopt_structures_guide[0].energy + 13.23 > max_guide_energy:
        max_guide_energy = subopt_structures_guide[0].energy + 13.23
    if subopt_structures_guide[0].energy + 13.23 < min_guide_energy:
        min_guide_energy = subopt_structures_guide[0].energy + 13.23
    prob_sub_target = math.exp((ensemble_energy_target - subopt_structures_target[0].energy) / kT)
    if subopt_structures_target[0].energy > max_target_energy:
        max_target_energy = subopt_structures_target[0].energy
    if subopt_structures_target[0].energy < min_target_energy:
        min_target_energy = subopt_structures_target[0].energy
    prob_sub_ssDNA_target_bh = math.exp((ensemble_energy_ssDNA_target_bh - subopt_structures_ssDNA_target_bh[0].energy) / kT)
    if subopt_structures_ssDNA_target_bh[0].energy > max_ssDNA_target_bh_energy:
        max_ssDNA_target_bh_energy = subopt_structures_ssDNA_target_bh[0].energy
    if subopt_structures_ssDNA_target_bh[0].energy < min_ssDNA_target_bh_energy:
        min_ssDNA_target_bh_energy = subopt_structures_ssDNA_target_bh[0].energy
        
    # Calculate/compare ensemble_energy_max_paired_guide and ensemble_energy_max_paired_target and ensemble_energy_max_paired_ssDNA_target_bh
    if str(subopt_structures_guide[0].structure)[0:len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_base_pairs(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_max_paired_guide = pfunc(strands=[max_paired_seq_guide, RNA_reverse_complement(max_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_paired_guide = partition_function_max_paired_guide[1]

    if str(subopt_structures_target[0].structure)[0:len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_base_pairs(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_max_paired_seq_target = RNA_to_DNA(max_paired_seq_target)
        partition_function_max_paired_target = pfunc(strands=[DNA_max_paired_seq_target, DNA_reverse_complement(DNA_max_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_paired_target = partition_function_max_paired_target[1]

    if str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = find_max_base_pairs(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_paired_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_max_paired_seq_ssDNA_target_bh = RNA_to_DNA(max_paired_seq_ssDNA_target_bh)
        partition_function_max_paired_ssDNA_target_bh = pfunc(strands=[DNA_max_paired_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_max_paired_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_max_paired_ssDNA_target_bh = partition_function_max_paired_ssDNA_target_bh[1]

    if ensemble_energy_max_paired_guide > max_guide_energy_conse:
        max_guide_energy_conse = ensemble_energy_max_paired_guide
    if ensemble_energy_max_paired_guide < min_guide_energy_conse:
        min_guide_energy_conse = ensemble_energy_max_paired_guide

    if ensemble_energy_max_paired_target > max_target_energy_conse:
        max_target_energy_conse = ensemble_energy_max_paired_target
    if ensemble_energy_max_paired_target < min_target_energy_conse:
        min_target_energy_conse = ensemble_energy_max_paired_target

    if ensemble_energy_max_paired_ssDNA_target_bh > max_ssDNA_target_bh_energy_conse:
        max_ssDNA_target_bh_energy_conse = ensemble_energy_max_paired_ssDNA_target_bh
    if ensemble_energy_max_paired_ssDNA_target_bh < min_ssDNA_target_bh_energy_conse:
        min_ssDNA_target_bh_energy_conse = ensemble_energy_max_paired_ssDNA_target_bh

    
    # Calculate/compare ensemble_energy_max_unpaired_guide and ensemble_energy_max_unpaired_target and ensemble_energy_max_unpaired_ssDNA_target_bh
    if is_all_parens(str(subopt_structures_guide[0].structure)[0:len_spacer]) == True:
        ensemble_energy_max_unpaired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_unpaired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_unpaired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_max_unpaired_guide = pfunc(strands=[max_unpaired_seq_guide, RNA_reverse_complement(max_unpaired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_unpaired_guide = partition_function_max_unpaired_guide[1]

    if is_all_parens(str(subopt_structures_target[0].structure)[0:len_spacer]) == True:
        ensemble_energy_max_unpaired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_unpaired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_unpaired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_max_unpaired_seq_target = RNA_to_DNA(max_unpaired_seq_target)
        partition_function_max_unpaired_target = pfunc(strands=[DNA_max_unpaired_seq_target, DNA_reverse_complement(DNA_max_unpaired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_unpaired_target = partition_function_max_unpaired_target[1]

    if is_all_parens(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer]) == True:
        ensemble_energy_max_unpaired_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = find_max_unpaired(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_unpaired_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_max_unpaired_seq_ssDNA_target_bh = RNA_to_DNA(max_unpaired_seq_ssDNA_target_bh)
        partition_function_max_unpaired_ssDNA_target_bh = pfunc(strands=[DNA_max_unpaired_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_max_unpaired_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_max_unpaired_ssDNA_target_bh = partition_function_max_unpaired_ssDNA_target_bh[1]

    if ensemble_energy_max_unpaired_guide > max_guide_energy_conse_unpaired:
        max_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide
    if ensemble_energy_max_unpaired_guide < min_guide_energy_conse_unpaired:
        min_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide

    if ensemble_energy_max_unpaired_target > max_target_energy_conse_unpaired:
        max_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target
    if ensemble_energy_max_unpaired_target < min_target_energy_conse_unpaired:
        min_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target

    if ensemble_energy_max_unpaired_ssDNA_target_bh > max_ssDNA_target_bh_energy_conse_unpaired:
        max_ssDNA_target_bh_energy_conse_unpaired = ensemble_energy_max_unpaired_ssDNA_target_bh
    if ensemble_energy_max_unpaired_ssDNA_target_bh < min_ssDNA_target_bh_energy_conse_unpaired:
        min_ssDNA_target_bh_energy_conse_unpaired = ensemble_energy_max_unpaired_ssDNA_target_bh



    # Calculate/compare ensemble_energy_PAM_proximal_overhang_guide and ensemble_energy_PAM_proximal_overhang_target and ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh
    if str(subopt_structures_guide[0].structure)[len_spacer-1] != '.':
        ensemble_energy_PAM_proximal_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_overhang(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_PAM_proximal_overhang_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_PAM_proximal_overhang_guide = pfunc(strands=[max_PAM_proximal_overhang_seq_guide, RNA_reverse_complement(max_PAM_proximal_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_PAM_proximal_overhang_guide = partition_function_PAM_proximal_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[len_spacer-1] != '.':
        ensemble_energy_PAM_proximal_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_overhang(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_PAM_proximal_overhang_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_PAM_proximal_overhang_seq_target = RNA_to_DNA(max_PAM_proximal_overhang_seq_target)
        partition_function_PAM_proximal_overhang_target = pfunc(strands=[DNA_PAM_proximal_overhang_seq_target, DNA_reverse_complement(DNA_PAM_proximal_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_PAM_proximal_overhang_target = partition_function_PAM_proximal_overhang_target[1]

    if str(subopt_structures_ssDNA_target_bh[0].structure)[0] != '.':
        ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = detect_5_overhang(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_PAM_proximal_overhang_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_PAM_proximal_overhang_seq_ssDNA_target_bh = RNA_to_DNA(max_PAM_proximal_overhang_seq_ssDNA_target_bh)
        partition_function_PAM_proximal_overhang_ssDNA_target_bh = pfunc(strands=[DNA_PAM_proximal_overhang_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_PAM_proximal_overhang_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh = partition_function_PAM_proximal_overhang_ssDNA_target_bh[1]


    if ensemble_energy_PAM_proximal_overhang_guide > max_guide_energy_PAM_proximal_overhang:
        max_guide_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_guide
    if ensemble_energy_PAM_proximal_overhang_guide < min_guide_energy_PAM_proximal_overhang:
        min_guide_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_guide

    if ensemble_energy_PAM_proximal_overhang_target > max_target_energy_PAM_proximal_overhang:
        max_target_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_target
    if ensemble_energy_PAM_proximal_overhang_target < min_target_energy_PAM_proximal_overhang:
        min_target_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_target

    if ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh > max_ssDNA_target_bh_energy_PAM_proximal_overhang:
        max_ssDNA_target_bh_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh
    if ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh < min_ssDNA_target_bh_energy_PAM_proximal_overhang:
        min_ssDNA_target_bh_energy_PAM_proximal_overhang = ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh


    # Calculate/compare ensemble_energy_PAM_distal_overhang_guide and ensemble_energy_PAM_distal_overhang_target and ensemble_energy_PAM_distal_overhang_ssDNA_target_bh
    if str(subopt_structures_guide[0].structure)[0] != '.':
        ensemble_energy_PAM_distal_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_overhang(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_PAM_distal_overhang_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_PAM_distal_overhang_guide = pfunc(strands=[max_PAM_distal_overhang_seq_guide, RNA_reverse_complement(max_PAM_distal_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_PAM_distal_overhang_guide = partition_function_PAM_distal_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[0] != '.':
        ensemble_energy_PAM_distal_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_overhang(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_PAM_distal_overhang_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_PAM_distal_overhang_seq_target = RNA_to_DNA(max_PAM_distal_overhang_seq_target)
        partition_function_PAM_distal_overhang_target = pfunc(strands=[DNA_PAM_distal_overhang_seq_target, DNA_reverse_complement(DNA_PAM_distal_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_PAM_distal_overhang_target = partition_function_PAM_distal_overhang_target[1]

    if str(subopt_structures_ssDNA_target_bh[0].structure)[len_spacer-1] != '.':
        ensemble_energy_PAM_distal_overhang_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = detect_3_overhang(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_PAM_distal_overhang_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_PAM_distal_overhang_seq_ssDNA_target_bh = RNA_to_DNA(max_PAM_distal_overhang_seq_ssDNA_target_bh)
        partition_function_PAM_distal_overhang_ssDNA_target_bh = pfunc(strands=[DNA_PAM_distal_overhang_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_PAM_distal_overhang_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_PAM_distal_overhang_ssDNA_target_bh = partition_function_PAM_distal_overhang_ssDNA_target_bh[1]


    if ensemble_energy_PAM_distal_overhang_guide > max_guide_energy_PAM_distal_overhang:
        max_guide_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_guide
    if ensemble_energy_PAM_distal_overhang_guide < min_guide_energy_PAM_distal_overhang:
        min_guide_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_guide

    if ensemble_energy_PAM_distal_overhang_target > max_target_energy_PAM_distal_overhang:
        max_target_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_target
    if ensemble_energy_PAM_distal_overhang_target < min_target_energy_PAM_distal_overhang:
        min_target_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_target

    if ensemble_energy_PAM_distal_overhang_ssDNA_target_bh > max_ssDNA_target_bh_energy_PAM_distal_overhang:
        max_ssDNA_target_bh_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_ssDNA_target_bh
    if ensemble_energy_PAM_distal_overhang_ssDNA_target_bh < min_ssDNA_target_bh_energy_PAM_distal_overhang:
        min_ssDNA_target_bh_energy_PAM_distal_overhang = ensemble_energy_PAM_distal_overhang_ssDNA_target_bh



    # Calculate/compare ensemble_energy_PAM_proximal_paired_guide and ensemble_energy_PAM_proximal_paired_target and ensemble_energy_PAM_proximal_paired_ssDNA_target_bh
    if str(subopt_structures_guide[0].structure)[len_spacer-1] == '.':
        ensemble_energy_PAM_proximal_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_paired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_PAM_proximal_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_PAM_proximal_paired_guide = pfunc(strands=[max_PAM_proximal_paired_seq_guide, RNA_reverse_complement(max_PAM_proximal_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_PAM_proximal_paired_guide = partition_function_PAM_proximal_paired_guide[1]

    if str(subopt_structures_target[0].structure)[len_spacer-1] == '.':
        ensemble_energy_PAM_proximal_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_paired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_PAM_proximal_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_PAM_proximal_paired_seq_target = RNA_to_DNA(max_PAM_proximal_paired_seq_target)
        partition_function_PAM_proximal_paired_target = pfunc(strands=[DNA_PAM_proximal_paired_seq_target, DNA_reverse_complement(DNA_PAM_proximal_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_PAM_proximal_paired_target = partition_function_PAM_proximal_paired_target[1]

    if str(subopt_structures_ssDNA_target_bh[0].structure)[0] == '.':
        ensemble_energy_PAM_proximal_paired_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = detect_5_paired(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_PAM_proximal_paired_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_PAM_proximal_paired_seq_ssDNA_target_bh = RNA_to_DNA(max_PAM_proximal_paired_seq_ssDNA_target_bh)
        partition_function_PAM_proximal_paired_ssDNA_target_bh = pfunc(strands=[DNA_PAM_proximal_paired_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_PAM_proximal_paired_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_PAM_proximal_paired_ssDNA_target_bh = partition_function_PAM_proximal_paired_ssDNA_target_bh[1]


    if ensemble_energy_PAM_proximal_paired_guide > max_guide_energy_PAM_proximal_paired:
        max_guide_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_guide
    if ensemble_energy_PAM_proximal_paired_guide < min_guide_energy_PAM_proximal_paired:
        min_guide_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_guide

    if ensemble_energy_PAM_proximal_paired_target > max_target_energy_PAM_proximal_paired:
        max_target_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_target
    if ensemble_energy_PAM_proximal_paired_target < min_target_energy_PAM_proximal_paired:
        min_target_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_target

    if ensemble_energy_PAM_proximal_paired_ssDNA_target_bh > max_ssDNA_target_bh_energy_PAM_proximal_paired:
        max_ssDNA_target_bh_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_ssDNA_target_bh
    if ensemble_energy_PAM_proximal_paired_ssDNA_target_bh < min_ssDNA_target_bh_energy_PAM_proximal_paired:
        min_ssDNA_target_bh_energy_PAM_proximal_paired = ensemble_energy_PAM_proximal_paired_ssDNA_target_bh

            

    # Calculate/compare ensemble_energy_PAM_distal_paired_guide and ensemble_energy_PAM_distal_paired_target and ensemble_energy_PAM_distal_paired_ssDNA_target_bh
    if str(subopt_structures_guide[0].structure)[0] == '.':
        ensemble_energy_PAM_distal_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_paired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_PAM_distal_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_PAM_distal_paired_guide = pfunc(strands=[max_PAM_distal_paired_seq_guide, RNA_reverse_complement(max_PAM_distal_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_PAM_distal_paired_guide = partition_function_PAM_distal_paired_guide[1]

    if str(subopt_structures_target[0].structure)[0] == '.':
        ensemble_energy_PAM_distal_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_paired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_PAM_distal_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_PAM_distal_paired_seq_target = RNA_to_DNA(max_PAM_distal_paired_seq_target)
        partition_function_PAM_distal_paired_target = pfunc(strands=[DNA_PAM_distal_paired_seq_target, DNA_reverse_complement(DNA_PAM_distal_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_PAM_distal_paired_target = partition_function_PAM_distal_paired_target[1]

    if str(subopt_structures_ssDNA_target_bh[0].structure)[len_spacer-1] == '.':
        ensemble_energy_PAM_distal_paired_ssDNA_target_bh = 0
    else:
        max_start_index_ssDNA_target_bh, max_length_ssDNA_target_bh = detect_3_paired(str(subopt_structures_ssDNA_target_bh[0].structure)[0:len_spacer])
        max_PAM_distal_paired_seq_ssDNA_target_bh = guide[max_start_index_ssDNA_target_bh:max_start_index_ssDNA_target_bh+max_length_ssDNA_target_bh]
        DNA_PAM_distal_paired_seq_ssDNA_target_bh = RNA_to_DNA(max_PAM_distal_paired_seq_ssDNA_target_bh)
        partition_function_PAM_distal_paired_ssDNA_target_bh = pfunc(strands=[DNA_PAM_distal_paired_seq_ssDNA_target_bh, DNA_reverse_complement(DNA_PAM_distal_paired_seq_ssDNA_target_bh)], model=my_model_DNA)
        ensemble_energy_PAM_distal_paired_ssDNA_target_bh = partition_function_PAM_distal_paired_ssDNA_target_bh[1]

    if ensemble_energy_PAM_distal_paired_guide > max_guide_energy_PAM_distal_paired:
        max_guide_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_guide
    if ensemble_energy_PAM_distal_paired_guide < min_guide_energy_PAM_distal_paired:
        min_guide_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_guide

    if ensemble_energy_PAM_distal_paired_target > max_target_energy_PAM_distal_paired:
        max_target_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_target
    if ensemble_energy_PAM_distal_paired_target < min_target_energy_PAM_distal_paired:
        min_target_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_target

    if ensemble_energy_PAM_distal_paired_ssDNA_target_bh > max_ssDNA_target_bh_energy_PAM_distal_paired:
        max_ssDNA_target_bh_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_ssDNA_target_bh
    if ensemble_energy_PAM_distal_paired_ssDNA_target_bh < min_ssDNA_target_bh_energy_PAM_distal_paired:
        min_ssDNA_target_bh_energy_PAM_distal_paired = ensemble_energy_PAM_distal_paired_ssDNA_target_bh


    # Calculate/compare target seed region free energy 
    seed_region = target_truncated[0:6]
    subopt_structures_seed = subopt(strands=[seed_region, DNA_reverse_complement(seed_region)], energy_gap=0.01, model=my_model_DNA)
    seed_energy = subopt_structures_seed[0].energy

    if seed_energy > max_seed_energy:
        max_seed_energy = seed_energy
    if seed_energy < min_seed_energy:
        min_seed_energy = seed_energy


    # Calculate/compare target middle region free energy 
    middle_region = target_truncated[6:13]
    subopt_structures_middle = subopt(strands=[middle_region, DNA_reverse_complement(middle_region)], energy_gap=0.01, model=my_model_DNA)
    middle_energy = subopt_structures_middle[0].energy

    if middle_energy > max_middle_energy:
        max_middle_energy = middle_energy
    if middle_energy < min_middle_energy:
        min_middle_energy = middle_energy


    # Calculate/compare target distal region free energy 
    distal_region = target_truncated[13:20]
    subopt_structures_distal = subopt(strands=[distal_region, DNA_reverse_complement(distal_region)], energy_gap=0.01, model=my_model_DNA)
    distal_energy = subopt_structures_distal[0].energy

    if distal_energy > max_distal_energy:
        max_distal_energy = distal_energy
    if distal_energy < min_distal_energy:
        min_distal_energy = distal_energy


    # Detect local energy for consecutive 3 bases in target
    min_ensemble_energy_selected_bases_3 = np.inf
    max_ensemble_energy_selected_bases_3 = -np.inf
    
    for k in range (0, len(target_truncated)-2):
        selected_bases_spacer = guide[k:3+k]
        selected_bases_target = target_truncated[17-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_3:
            max_ensemble_energy_selected_bases_3 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_3:
            min_ensemble_energy_selected_bases_3 = ensemble_energy_selected_bases

    array_max_ensemble_energy_selected_bases_3.append(max_ensemble_energy_selected_bases_3)
    array_min_ensemble_energy_selected_bases_3.append(min_ensemble_energy_selected_bases_3)


    # Detect local energy for consecutive 4 bases in target
    min_ensemble_energy_selected_bases_4 = np.inf
    max_ensemble_energy_selected_bases_4 = -np.inf
    
    for k in range (0, len(target_truncated)-3):
        selected_bases_spacer = guide[k:4+k]
        selected_bases_target = target_truncated[16-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_4:
            max_ensemble_energy_selected_bases_4 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_4:
            min_ensemble_energy_selected_bases_4 = ensemble_energy_selected_bases

    array_max_ensemble_energy_selected_bases_4.append(max_ensemble_energy_selected_bases_4)
    array_min_ensemble_energy_selected_bases_4.append(min_ensemble_energy_selected_bases_4)

    # Detect local energy for consecutive 5 bases in target
    min_ensemble_energy_selected_bases_5 = np.inf
    max_ensemble_energy_selected_bases_5 = -np.inf

    
    for k in range (0, len(target_truncated)-4):
        selected_bases_spacer = guide[k:5+k]
        selected_bases_target = target_truncated[15-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_5:
            max_ensemble_energy_selected_bases_5 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_5:
            min_ensemble_energy_selected_bases_5 = ensemble_energy_selected_bases

    array_max_ensemble_energy_selected_bases_5.append(max_ensemble_energy_selected_bases_5)
    array_min_ensemble_energy_selected_bases_5.append(min_ensemble_energy_selected_bases_5)

    # Detect local energy for consecutive 6 bases in target
    min_ensemble_energy_selected_bases_6 = np.inf
    max_ensemble_energy_selected_bases_6 = -np.inf

    
    for k in range (0, len(target_truncated)-5):
        selected_bases_spacer = guide[k:6+k]
        selected_bases_target = target_truncated[14-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_6:
            max_ensemble_energy_selected_bases_6 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_6:
            min_ensemble_energy_selected_bases_6 = ensemble_energy_selected_bases

    array_max_ensemble_energy_selected_bases_6.append(max_ensemble_energy_selected_bases_6)
    array_min_ensemble_energy_selected_bases_6.append(min_ensemble_energy_selected_bases_6)

    # Detect local energy for consecutive 7 bases in target
    min_ensemble_energy_selected_bases_7 = np.inf
    max_ensemble_energy_selected_bases_7 = -np.inf
    
    for k in range (0, len(target_truncated)-6):
        selected_bases_spacer = guide[k:7+k]
        selected_bases_target = target_truncated[13-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_7:
            max_ensemble_energy_selected_bases_7 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_7:
            min_ensemble_energy_selected_bases_7 = ensemble_energy_selected_bases


    array_max_ensemble_energy_selected_bases_7.append(max_ensemble_energy_selected_bases_7)
    array_min_ensemble_energy_selected_bases_7.append(min_ensemble_energy_selected_bases_7)

    # Detect local energy for consecutive 8 bases in target
    min_ensemble_energy_selected_bases_8 = np.inf
    max_ensemble_energy_selected_bases_8 = -np.inf
    
    for k in range (0, len(target_truncated)-7):
        selected_bases_spacer = guide[k:8+k]
        selected_bases_target = target_truncated[12-k:20-k]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        if ensemble_energy_selected_bases > max_ensemble_energy_selected_bases_8:
            max_ensemble_energy_selected_bases_8 = ensemble_energy_selected_bases
        if ensemble_energy_selected_bases < min_ensemble_energy_selected_bases_8:
            min_ensemble_energy_selected_bases_8 = ensemble_energy_selected_bases

    array_max_ensemble_energy_selected_bases_8.append(max_ensemble_energy_selected_bases_8)
    array_min_ensemble_energy_selected_bases_8.append(min_ensemble_energy_selected_bases_8)


    # Compute the energy of every consecutive 4 base pairs in spacer-target duplex
    # This aims to quantify the effect of homopolymers by detecting local high and low energy regions 
    four_base_consecutive_energy_array = [0 for _ in range(17)]

    d = 4
    for s in range (0, 17):
        selected_bases_spacer = guide[20-s-d:20-s]
        selected_bases_target = target_truncated[s:s+d]
        partition_function_selected_bases = pfunc(strands=[selected_bases_spacer, selected_bases_target], model=my_model_RNA)
        ensemble_energy_selected_bases = partition_function_selected_bases[1]
        four_base_consecutive_energy_array[s] = ensemble_energy_selected_bases


    # Calculate ensemble defect of crRNA-target duplex towards its target structure, which is the fully binding state of crRNA and target duplex   
    ensemble_defect_duplex_array = [0 for _ in range(19)]
    for d in range (0, 17):
        for s in range (2, 5):
            guide_segment = guide[20-d-s:20-d]
            target_segment = target_truncated[d:d+s]
            duplex_fully_binding = "(" * s + "+" + ")" * s
            ensemble_defect_duplex_ah = defect(strands=[guide_segment, target_segment], structure=duplex_fully_binding, model=my_model_RNA)
            ensemble_defect_duplex_array[d] += ensemble_defect_duplex_ah

    for d in range (17, 19):
        for s in range (2, 21-d):
            guide_segment = guide[20-d-s:20-d]
            target_segment = target_truncated[d:d+s]
            duplex_fully_binding = "(" * s + "+" + ")" * s
            ensemble_defect_duplex_ah = defect(strands=[guide_segment, target_segment], structure=duplex_fully_binding, model=my_model_RNA)
            ensemble_defect_duplex_array[d] += ensemble_defect_duplex_ah



    # Calculate the energy of base pairings at incorrect positions of crRNA-target duplex 
    ensemble_defect_energy_duplex_array = [0 for _ in range(18)]

    s = 2
    duplex_fully_binding = "(" * s + "+" + ")" * s
    for guide_start in range (0, 16):
        guide_segment = guide[20-guide_start-s:20-guide_start]
        for target_start in range (guide_start+1, guide_start+4):
            target_segment = target_truncated[target_start:target_start+s]    
            subopt_structures_pairing = subopt(strands=[guide_segment, target_segment], energy_gap=0.01, model=my_model_RNA)
            # Compute suboptimal structures and energy of base pairings at incorrect positions
            if subopt_structures_pairing != []:
                if str(subopt_structures_pairing[0].structure) == duplex_fully_binding:
                    if subopt_structures_pairing[0].energy >= 0:
                        ensemble_defect_energy_duplex_array[guide_start] += subopt_structures_pairing[0].energy
                    else:
                        ensemble_defect_energy_duplex_array[guide_start] -= subopt_structures_pairing[0].energy

    for guide_start in range (16, 18):
        guide_segment = guide[20-guide_start-s:20-guide_start]
        for target_start in range (guide_start+1, 20):
            target_segment = target_truncated[target_start:target_start+s]    
            subopt_structures_pairing = subopt(strands=[guide_segment, target_segment], energy_gap=0.01, model=my_model_RNA)
            # Compute suboptimal structures and energy of base pairings at incorrect positions
            if subopt_structures_pairing != []:
                if str(subopt_structures_pairing[0].structure) == duplex_fully_binding:
                    if subopt_structures_pairing[0].energy >= 0:
                        ensemble_defect_energy_duplex_array[guide_start] += subopt_structures_pairing[0].energy
                    else:
                        ensemble_defect_energy_duplex_array[guide_start] -= subopt_structures_pairing[0].energy


    


    to_be_added = [normalize_guide(subopt_structures_guide[0].energy + 13.23)]
    to_be_added.append(normalize_guide_conse(ensemble_energy_max_paired_guide))
    to_be_added.append(normalize_guide_conse_unpaired(ensemble_energy_max_unpaired_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_PAM_proximal_overhang_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_PAM_distal_overhang_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_PAM_proximal_paired_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_PAM_distal_paired_guide))
    
    to_be_added.append(normalize_target(subopt_structures_target[0].energy))
    to_be_added.append(normalize_target_conse(ensemble_energy_max_paired_target))
    to_be_added.append(normalize_target_conse_unpaired(ensemble_energy_max_unpaired_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_PAM_proximal_overhang_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_PAM_distal_overhang_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_PAM_proximal_paired_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_PAM_distal_paired_target))

    to_be_added.append(normalize_ssDNA_target_bh(subopt_structures_ssDNA_target_bh[0].energy))
    to_be_added.append(normalize_ssDNA_target_bh_conse(ensemble_energy_max_paired_ssDNA_target_bh))
    to_be_added.append(normalize_ssDNA_target_bh_conse_unpaired(ensemble_energy_max_unpaired_ssDNA_target_bh))
    to_be_added.append(normalize_ssDNA_target_bh_overhang(ensemble_energy_PAM_proximal_overhang_ssDNA_target_bh))
    to_be_added.append(normalize_ssDNA_target_bh_overhang(ensemble_energy_PAM_distal_overhang_ssDNA_target_bh))
    to_be_added.append(normalize_ssDNA_target_bh_paired(ensemble_energy_PAM_proximal_paired_ssDNA_target_bh))
    to_be_added.append(normalize_ssDNA_target_bh_paired(ensemble_energy_PAM_distal_paired_ssDNA_target_bh))
    
    to_be_added.append(normalize_seed(seed_energy))
    to_be_added.append(normalize_middle(middle_energy))
    to_be_added.append(normalize_distal(distal_energy))


    to_be_added.append(normalize_target_conse_3(max_ensemble_energy_selected_bases_3))
    to_be_added.append(normalize_target_conse_3(min_ensemble_energy_selected_bases_3))
    to_be_added.append(normalize_target_conse_4(max_ensemble_energy_selected_bases_4))
    to_be_added.append(normalize_target_conse_4(min_ensemble_energy_selected_bases_4))
    to_be_added.append(normalize_target_conse_5(max_ensemble_energy_selected_bases_5))
    to_be_added.append(normalize_target_conse_5(min_ensemble_energy_selected_bases_5))
    to_be_added.append(normalize_target_conse_6(max_ensemble_energy_selected_bases_6))
    to_be_added.append(normalize_target_conse_6(min_ensemble_energy_selected_bases_6))
    to_be_added.append(normalize_target_conse_7(max_ensemble_energy_selected_bases_7))
    to_be_added.append(normalize_target_conse_7(min_ensemble_energy_selected_bases_7))
    to_be_added.append(normalize_target_conse_8(max_ensemble_energy_selected_bases_8))
    to_be_added.append(normalize_target_conse_8(min_ensemble_energy_selected_bases_8))


    for s in range (0, 17):
        to_be_added.append(normalize_target_conse_4(four_base_consecutive_energy_array[s]))

    for s in range (0, 10):
        to_be_added.append(0)

    for d in range (0, 19):
        to_be_added.append(ensemble_defect_duplex_array[d])

    for s in range (0, 10):
        to_be_added.append(0)

    for s in range (0, 18):
        to_be_added.append(ensemble_defect_energy_duplex_array[s])

    for s in range (0, 10):
        to_be_added.append(0)

    struct_prob_array_unit.append(to_be_added)

    struct_prob_array_to_append = [struct_prob_array_unit]
    struct_prob_array.append(struct_prob_array_to_append)

struct_prob_array = list(itertools.chain.from_iterable(struct_prob_array))

# Open a file in write mode
with open('Feature_guide_target_complete_value_features_120D_all_positive_HTCas9_unique_KD_reduced_feature_model.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in struct_prob_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')
        

print('-------------')
 
print(min_guide_energy, max_guide_energy, min_target_energy, max_target_energy, min_ssDNA_target_bh_energy, max_ssDNA_target_bh_energy)
print(min_guide_energy_conse, max_guide_energy_conse, min_target_energy_conse, max_target_energy_conse, min_ssDNA_target_bh_energy_conse, max_ssDNA_target_bh_energy_conse)
print(min_guide_energy_conse_unpaired, max_guide_energy_conse_unpaired, min_target_energy_conse_unpaired, max_target_energy_conse_unpaired, min_ssDNA_target_bh_energy_conse_unpaired, max_ssDNA_target_bh_energy_conse_unpaired)
print(min_guide_energy_PAM_proximal_overhang, max_guide_energy_PAM_proximal_overhang, min_target_energy_PAM_proximal_overhang, max_target_energy_PAM_proximal_overhang, min_ssDNA_target_bh_energy_PAM_proximal_overhang, max_ssDNA_target_bh_energy_PAM_proximal_overhang)
print(min_guide_energy_PAM_distal_overhang, max_guide_energy_PAM_distal_overhang, min_target_energy_PAM_distal_overhang, max_target_energy_PAM_distal_overhang, min_ssDNA_target_bh_energy_PAM_distal_overhang, max_ssDNA_target_bh_energy_PAM_distal_overhang)
print(min_guide_energy_PAM_proximal_paired, max_guide_energy_PAM_proximal_paired, min_target_energy_PAM_proximal_paired, max_target_energy_PAM_proximal_paired, min_ssDNA_target_bh_energy_PAM_proximal_paired, max_ssDNA_target_bh_energy_PAM_proximal_paired)
print(min_guide_energy_PAM_distal_paired, max_guide_energy_PAM_distal_paired, min_target_energy_PAM_distal_paired, max_target_energy_PAM_distal_paired, min_ssDNA_target_bh_energy_PAM_distal_paired, max_ssDNA_target_bh_energy_PAM_distal_paired)

print(max_seed_energy, min_seed_energy)
print(max_middle_energy, min_middle_energy)
print(max_distal_energy, min_distal_energy)